# Object-Oriented Programming (OOPS) in Python

Object-Oriented Programming is a way of writing code where we organize everything into **objects** and **classes**. Think of it like building with Lego blocks - each block (object) has its own properties (like color and size) and can do specific things (like snap together). This makes our code more organized, reusable, and easier to understand.

**Key Concepts:**
- **Class**: A blueprint or template for creating objects
- **Object**: An actual thing created from a class
- **Methods**: Actions or things an object can do
- **Attributes**: Properties or characteristics of an object
- **Inheritance**: A class can inherit attributes and methods from another class, promoting code reuse
- **Polymorphism**: Methods with the same name can work differently in different classes
- **Encapsulation**: Bundling attributes and methods together while hiding internal details from the outside world
- **Abstraction**: Showing only essential features while hiding complex implementation details


### Classes

In [ ]:
class UserDetails:
    species = "Human"  # Class attribute

    def __init__(self, name, phone_number):
        self.name = name                  # Instance attribute
        self.phone_number = phone_number  # Instance attribute
        
# Creating an object of the UserDetails class
user = UserDetails("Buddy", "123-456-7890")
print(f"Name: {user.name}, Phone Number: {user.phone_number}, Species: {user.species}")

# Converting object attributes to JSON format
import json
print(json.dumps(user.__dict__, indent=4))

**Note**: As demonstrated in the above class, we manually defined the constructor and attributes using self. This process can be simplified by utilizing the Pydantic BaseModel, as shown below.

#### Defining Pydantic Models

In [ ]:
import uuid
from pydantic import BaseModel, Field
from datetime import datetime, timezone

class AdditionalDetails(BaseModel):
    email: str = Field(default=None, description="The email address of the user")
    age: int = Field(default=None, description="The age of the user")

class UserAddress(BaseModel):
    street: str = Field(default=None, description="The street address of the user")
    city: str = Field(default=None, description="The city of the user")
    state: str = Field(default=None, description="The state of the user")
    country: str = Field(default=None, description="The country of the user")
    pin_code: int = Field(default=None, description="The pin code of the user's address")

class UserDetails(BaseModel):
    id: str = Field(default_factory=lambda: str(uuid.uuid4()), description="The unique identifier for the user")
    name: str = Field(default=None, description="The full name of the user")
    phone_number: str = Field(default=None, description="The phone number of the user")
    additional_details: AdditionalDetails = Field(default_factory=AdditionalDetails, description="Additional details about the user")
    address: list[UserAddress] = Field(default_factory=list, description="List of address associated with the user")
    created_at: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat(), description="The timestamp when the user was created")
    updated_at: str = Field(default_factory=lambda: datetime.now(timezone.utc).isoformat(), description="The timestamp when the user was last updated")

In [ ]:
user_details = UserDetails(name="John Doe", phone_number="+1234567890")
user_details.additional_details = AdditionalDetails(email="john.doe@example.com", age=30)
user_details.address.append(UserAddress(street="123 Main St", city="Anytown", state="Anystate", country="Anycountry", pin_code=123456))
user_details.address.append(UserAddress(street="456 Side St", city="Othertown", state="Otherstate", country="Othercountry", pin_code=654321))

# Spread the UserDetails model into the JSON dictionary.
# In Pydantic v2, BaseModel provides model_dump_json() to serialize a model to JSON. 
print(user_details.model_dump_json(indent=4))

#### Creating Model Instance from JSON

In [ ]:
jane_user_details = {
    "id": str(uuid.uuid4()),
    "name": "Jane Smith",
    "phone_number": "+0987654321",
    "additional_details": {
        "email": "jane.smith@example.com",
        "age": 28
    },
    "address": [
        {
            "street": "789 Central Ave",
            "city": "Middletown",
            "state": "Middlestate",
            "country": "Middlecountry",
            "pin_code": 654321
        }
    ],
    "created_at": "2024-06-01T12:00:00.000000+00:00",
    "updated_at": "2024-06-01T12:30:00.000000+00:00",
}

# Spread the JSON dictionary into the UserDetails model
jane_user_model = UserDetails(**jane_user_details)

print(jane_user_model)

#### Using TypedDict (Not related to Pydantic)

In [ ]:
from typing import TypedDict

class CompanyDetails(TypedDict):
    id: str
    company_name: str
    employee_count: int
    founded_year: int
    
company_deatails = CompanyDetails(id=str(uuid.uuid4()), company_name="Tech Solutions")
company_deatails["employee_count"] = 150
company_deatails["founded_year"] = 2010

print(company_deatails)

### Inheritance

Inheritance allows a class (child class) to acquire attributes and methods of another class (parent class). It supports hierarchical classification and promotes code reuse.

In [ ]:
# Example of Multiple Inheritance
# A child class inherits from multiple parent classes

class Vehicle:
    def __init__(self, brand: str):
        self.brand = brand
    
    def get_brand(self):
        return f"Brand: {self.brand}"

class Engine:
    def __init__(self, engine_type: str, is_petrol: bool):
        self.engine_type = engine_type
        self.is_petrol = is_petrol
    
    def get_engine_info(self):
        return f"Engine: {self.engine_type}, Petrol: {self.is_petrol}"

# Car inherits from both Vehicle and Engine (Multiple Inheritance)
class Car(Vehicle, Engine):
    def __init__(self, brand: str, engine_type: str, is_petrol: bool, color: str):
        Vehicle.__init__(self, brand)
        Engine.__init__(self, engine_type, is_petrol)
        self.color = color
    
    def get_car_details(self):
        return f"{self.get_brand()}, Color: {self.color}, {self.get_engine_info()}"

# Creating an instance of Car
car = Car(brand="Toyota", engine_type="1.8L Hybrid", is_petrol=True, color="Silver")
print(car.get_car_details())

### Polymorphism

It allows methods with the same name to work differently. There are 2 concepts in Polymorphism - Method Overloading & Method Overriding

In [ ]:
# Method Overloading - Having multiple methods with the same name but different parameters.

# Note on Python: Python does not support traditional method overloading like Java or C++.
# If you define two methods with the same name, the second one replaces the first. 
# However, we achieve similar behavior using default arguments or variable-length arguments (*args).

class Calculator:
    def add(self, *args):
        return sum(args)

calc = Calculator()
print(calc.add(5, 10))       # Two arguments
print(calc.add(5, 10, 15))   # Three arguments
print(calc.add(1, 2, 3, 4))  # Any number of arguments

In [ ]:
# Method Overriding - Providing a new implementation for a parent class method in the child class.

class Animal:
    def speak(self):
        return "I am an animal."

# "overrides" the speak method.
class Dog(Animal):
    def speak(self):
        return "Woof!"

print(Dog().speak())

### Encapsulation

**Encapsulation** is the practice of bundling attributes and methods together within a class, and hiding the internal details from the outside world.

In [ ]:
class BankAccount:
    def __init__(self, owner, balance):
        self.owner = owner  # Public attribute
        self._interest_rate = 0.05  # Protected attribute
        self.__balance = balance  # Private attribute
    
    # Public method to safely access balance
    def get_balance(self):
        return self.__balance
    
    # Public method to safely modify balance
    def deposit(self, amount):
        if amount > 0:
            self.__balance += amount
            return f"Deposited ${amount}"
        return "Invalid amount"
    
    # Public method to calculate interest using protected attribute
    def calculate_interest(self):
        return self.__balance * self._interest_rate

# Creating an account
account = BankAccount("Alice", 1000)
print(account.get_balance())  # 1000
print(account.deposit(500))   # Deposited $500

# Accessing public attribute (works fine)
print(f"Owner: {account.owner}")

# Accessing protected attribute (works but not recommended)
print(f"Interest Rate: {account._interest_rate}")  # 0.05

# Accessing private attribute (won't work directly)
# print(account.__balance)  # This will raise an AttributeError!

# But we can access it through public methods
print(f"Interest earned: ${account.calculate_interest()}")  # $75.0

# Attempting to modify protected attribute (possible but not recommended)
account._interest_rate = 0.10
print(f"New Interest: ${account.calculate_interest()}")  # $150.0

### Abstraction

**Abstraction** means showing only the essential features while hiding the complex internal details.

**Think of it like this:**
- When you drive a car, you just press the gas pedal (simple!)
- You don't need to know about fuel injection, pistons, or combustion happening inside the engine (complex!)
- The pedal is the **simple interface**, and all the engine stuff is **hidden complexity**

**Important:** "Hiding" doesn't mean you can't see the code. It means users of your class don't need to **understand** or **call** the complex internal methods. They only use the simple interface.

#### Simple Example: Abstract Class vs Interface in Python

**Python doesn't have a separate "interface" keyword** like Java or C#. Instead, we use abstract classes for both purposes:

1. **Interface-like (Pure Abstract Class):** Only has abstract methods (no implementation)
2. **Abstract Class:** Has both abstract methods AND concrete methods (with implementation)

In [ ]:
from abc import ABC, abstractmethod

# INTERFACE-LIKE: Only abstract methods (defines "what" but not "how")
class Drawable(ABC):
    @abstractmethod
    def draw(self):
        pass

# ABSTRACT CLASS: Mix of abstract and concrete methods
class Shape(ABC):
    def __init__(self, color):
        self.color = color  # Concrete implementation
    
    @abstractmethod
    def area(self):
        pass  # Child classes must implement this
    
    def describe(self):  # Concrete method - already implemented
        return f"A {self.color} shape"

In [ ]:
# Now let's create a Circle class that uses both
class Circle(Shape, Drawable):
    def __init__(self, color, radius):
        Shape.__init__(self, color)  # Calls Shape's __init__
        self.radius = radius
    
    # MUST implement this (from Shape)
    def area(self):
        return 3.14 * self.radius ** 2
    
    # MUST implement this (from Drawable)
    def draw(self):
        return f"Drawing a {self.color} circle"

# Using the Circle class
circle = Circle("red", 5)
print(circle.describe())  # Uses Shape's concrete method
print(circle.draw())      # Uses Circle's implementation
print(f"Area: {circle.area()}")  # Uses Circle's implementation

**What did we learn?**
- `Drawable` is interface-like → just says "you must have a draw method"
- `Shape` is an abstract class → gives us `color` and `describe()` for free, but forces us to write our own `area()`
- `Circle` gets code from `Shape` and fulfills the contract from `Drawable`

#### Real-World Example: Payment Processing System

This example shows TRUE abstraction - hiding complex details behind a simple interface.

In [ ]:
# Abstract class that defines the simple interface
class PaymentProcessor(ABC):
    @abstractmethod
    def process_payment(self, amount):
        """Users only see this simple method"""
        pass

# Credit Card implementation with HIDDEN complexity
class CreditCardProcessor(PaymentProcessor):
    def process_payment(self, amount):
        # All these complex steps are HIDDEN from the user
        self._validate_card()
        self._check_fraud()
        self._connect_to_bank()
        self._encrypt_data()
        self._send_to_gateway()
        return f"✓ Processed ${amount} via Credit Card"
    
    # These methods are INTERNAL - users don't need to know about them
    def _validate_card(self):
        """Check if card number is valid"""
        print("  → Validating card...")
    
    def _check_fraud(self):
        """Run fraud detection algorithms"""
        print("  → Checking for fraud...")
    
    def _connect_to_bank(self):
        """Connect to bank's API"""
        print("  → Connecting to bank...")
    
    def _encrypt_data(self):
        """Encrypt sensitive information"""
        print("  → Encrypting data...")
    
    def _send_to_gateway(self):
        """Send to payment gateway"""
        print("  → Sending to payment gateway...")

# PayPal implementation (different internal steps, same simple interface)
class PayPalProcessor(PaymentProcessor):
    def process_payment(self, amount):
        self._authenticate_paypal()
        self._verify_account()
        self._process_transaction()
        return f"✓ Processed ${amount} via PayPal"
    
    def _authenticate_paypal(self):
        print("  → Authenticating PayPal account...")
    
    def _verify_account(self):
        print("  → Verifying account balance...")
    
    def _process_transaction(self):
        print("  → Processing PayPal transaction...")

In [ ]:
# User's simple experience (THIS IS ABSTRACTION!)
print("=== User makes a payment ===")
processor = CreditCardProcessor()
result = processor.process_payment(100)
print(result)

print("\n=== User switches to PayPal ===")
processor = PayPalProcessor()
result = processor.process_payment(50)
print(result)

# The user only calls ONE simple method: process_payment()
# They don't need to know about validation, encryption, fraud checks, etc.
# That's the power of abstraction!

### OOPS Real world example - Hotel Booking System

5. Hotel Booking System
- Classes: Room, Booking, Guest, Hotel
- Inheritance: SingleRoom, DoubleRoom, Suite (inherit from Room)
- Polymorphism: Different room types have different pricing logic
- Methods: Check availability, Book room, Cancel booking

In [ ]:
from abc import ABC, abstractmethod

class Room:
    def __init__(self, name, description):
        name: str = name
        description: str = description
    